In [8]:
# generate_dataset_from_seeds.py
import os, cv2, numpy as np, random, shutil, uuid, zipfile
from pathlib import Path

SEEDS_DIR = "seeds"            # put seed images in seeds/<brand>/
OUT_DIR  = "dataset_generated" # output: dataset_generated/real & dataset_generated/fake
TARGET_PER_BRAND = 50          # total real images per brand to generate
FAKE_VARIANTS_PER_REAL = 1     # how many fake variants per real

os.makedirs(OUT_DIR, exist_ok=True)
real_out = os.path.join(OUT_DIR, "real")
fake_out = os.path.join(OUT_DIR, "fake")
os.makedirs(real_out, exist_ok=True)
os.makedirs(fake_out, exist_ok=True)

def random_brightness_contrast(img):
    alpha = 0.7 + np.random.rand()*0.8   # 0.7-1.5
    beta = int((np.random.rand()-0.5)*80) # -40..+40
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def add_noise(img):
    h,w,ch = img.shape
    gauss = np.random.normal(0, 12, (h,w,ch)).astype('uint8')
    return cv2.add(img, gauss)

def jpeg_compress(img, q=30):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), q]
    _, encimg = cv2.imencode('.jpg', img, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

def motion_blur(img, k=15):
    k = random.choice([7,11,15])
    kernel = np.zeros((k,k))
    kernel[int((k-1)/2), :] = np.ones(k)
    kernel = kernel / k
    return cv2.filter2D(img, -1, kernel)

def add_scratch(img):
    out = img.copy()
    h,w,_ = img.shape
    for _ in range(random.randint(1,3)):
        x1 = random.randint(0,w-1); y1 = random.randint(0,h-1)
        x2 = min(w-1, x1+random.randint(20, int(w*0.5)))
        y2 = min(h-1, y1+random.randint(-20,20))
        color = (random.randint(150,255),)*3
        thickness = random.randint(1,3)
        cv2.line(out, (x1,y1), (x2,y2), color, thickness)
    return out

def warp_perspective(img):
    h,w = img.shape[:2]
    max_shift = int(0.08*min(h,w))
    pts1 = np.float32([[0,0],[w,0],[0,h]])
    pts2 = np.float32([[random.randint(0,max_shift), random.randint(0,max_shift)],
                       [w-random.randint(0,max_shift), random.randint(0,max_shift)],
                       [random.randint(0,max_shift), h-random.randint(0,max_shift)]])
    M = cv2.getAffineTransform(pts1, pts2)
    return cv2.warpAffine(img, M, (w,h), borderMode=cv2.BORDER_REFLECT)

def generate_real_variants(img, n):
    outs = []
    for i in range(n):
        out = img.copy()
        # Random crop + resize
        h,w = out.shape[:2]
        cx = random.uniform(0.9,1.0); cy = random.uniform(0.9,1.0)
        nw, nh = int(w*cx), int(h*cy)
        x = random.randint(0, max(0,w-nw))
        y = random.randint(0, max(0,h-nh))
        out = out[y:y+nh, x:x+nw]
        out = cv2.resize(out, (int(w), int(h)))
        out = random_brightness_contrast(out)
        if random.random() < 0.25:
            out = add_noise(out)
        if random.random() < 0.2:
            out = jpeg_compress(out, q=random.randint(60,95))
        if random.random() < 0.15:
            out = warp_perspective(out)
        outs.append(out)
    return outs

def generate_fake_variant(img):
    out = img.copy()
    # heavy blur or motion blur
    if random.random() < 0.6:
        out = cv2.GaussianBlur(out, (random.choice([3,5,7,9]),)*2, 0)
    if random.random() < 0.6:
        out = motion_blur(out)
    if random.random() < 0.6:
        out = add_scratch(out)
    if random.random() < 0.6:
        out = jpeg_compress(out, q=random.randint(20,50))
    if random.random() < 0.5:
        out = add_noise(out)
    if random.random() < 0.3:
        out = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
        out = cv2.cvtColor(out, cv2.COLOR_GRAY2BGR)
    # random crop simulate partial occlusion
    h,w = out.shape[:2]
    if random.random() < 0.3:
        x = random.randint(0, int(w*0.3)); y = random.randint(0, int(h*0.3))
        out[y:y+int(h*0.4), x:x+int(w*0.4)] = (random.randint(0,30),)*3
    return out

# iterate brands
for brand in os.listdir(SEEDS_DIR):
    brand_dir = os.path.join(SEEDS_DIR, brand)
    if not os.path.isdir(brand_dir): continue
    files = [f for f in os.listdir(brand_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if not files: continue
    print("Processing brand:", brand)
    # ensure at least one seed to copy as real
    idx = 0
    for seed_file in files:
        seed_path = os.path.join(brand_dir, seed_file)
        img = cv2.imread(seed_path)
        if img is None: continue
        # generate real variants
        num_needed = TARGET_PER_BRAND
        reals = generate_real_variants(img, num_needed//len(files) + 1)
        for r in reals[:max(1, num_needed//len(files))]:
            fname = f"{brand}_real_{uuid.uuid4().hex[:8]}.jpg"
            cv2.imwrite(os.path.join(real_out, fname), r)
            idx += 1
            if idx >= TARGET_PER_BRAND: break
        # generate fake variants for each real
        for _ in range(FAKE_VARIANTS_PER_REAL * (num_needed//len(files) + 1)):
            fv = generate_fake_variant(img)
            fname = f"{brand}_fake_{uuid.uuid4().hex[:8]}.jpg"
            cv2.imwrite(os.path.join(fake_out, fname), fv)
        if idx >= TARGET_PER_BRAND: break

# Zip the results
zip_name = "dataset_generated.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root,_,files in os.walk(OUT_DIR):
        for f in files:
            zf.write(os.path.join(root,f), os.path.relpath(os.path.join(root,f), OUT_DIR))

print("Done. Output folder:", OUT_DIR, "Zip:", zip_name)


Done. Output folder: dataset_generated Zip: dataset_generated.zip
